# **1. Problem Statement**


## 1.1 Business Context


A sales forecast is a prediction of future sales revenue based on historical data, industry trends, and the status of the current sales pipeline. Businesses use the sales forecast to estimate weekly, monthly, quarterly, and annual sales totals. A company needs to make an accurate sales forecast as it adds value across an organization and helps the different verticals to chalk out their future course of action.

Forecasting helps an organization plan its sales operations by region and provides valuable insights to the supply chain team regarding the procurement of goods and materials. An accurate sales forecast process has many benefits which include improved decision-making about the future and reduction of sales pipeline and forecast risks. Moreover, it helps to reduce the time spent in planning territory coverage and establish benchmarks that can be used to assess trends in the future.

## 1.2 Objective


SuperKart is a retail chain operating supermarkets and food marts across various tier cities, offering a wide range of products. To optimize its inventory management and make informed decisions around regional sales strategies, SuperKart wants to accurately forecast the sales revenue of its outlets for the upcoming quarter.

To operationalize these insights at scale, the company has partnered with a data science firm to develop and deploy a robust forecasting solution that can be integrated into SuperKart’s decision-making systems and used across its network of stores.

## 1.3 Data Description


The data contains the different attributes of the various products and stores. The detailed data dictionary is given below.

- **Product_Id** - unique identifier of each product, each identifier having two letters at the beginning followed by a number.
- **Product_Weight** - weight of each product
- **Product_Sugar_Content** - sugar content of each product like low sugar, regular and no sugar
- **Product_Allocated_Area** - ratio of the allocated display area of each product to the total display area of all the products in a store
- **Product_Type** - broad category for each product
- **Product_MRP** - maximum retail price of each product
- **Store_Id** - unique identifier of each store
- **Store_Establishment_Year** - year in which the store was established
- **Store_Size** - size of the store depending on sq. feet like high, medium and low
- **Store_Location_City_Type** - type of city in which the store is located like Tier 1, Tier 2 and Tier 3.
- **Store_Type** - type of store depending on the products that are being sold there
- **Product_Store_Sales_Total** - total revenue generated by the sale of that particular product in that particular store


# **2. Installing and Importing the necessary libraries**


In [1]:
%pip install pandas scikit-learn xgboost huggingface_hub joblib


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from huggingface_hub import HfApi, login
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score


# **3. Data Loading and Hugging Face Registration**


In [3]:
# Authenticate with Hugging Face using the provided token
print('Logging into Hugging Face...')
import os
token = os.getenv('HF_TOKEN')
if token:
    login(token)
else:
    from huggingface_hub import notebook_login
    notebook_login()


Logging into Hugging Face...


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
# Register the raw dataset to the Hugging Face dataset space first
api = HfApi()
print('Uploading raw dataset to Hugging Face dataset space...')
api.create_repo(repo_id='pvinayv/superkart-dataset', repo_type='dataset', exist_ok=True)
api.upload_file(
    path_or_fileobj='data/SuperKart.csv',
    path_in_repo='data/SuperKart.csv',
    repo_id='pvinayv/superkart-dataset',
    repo_type='dataset'
)
print('Raw dataset registered on Hugging Face dataset space.')


Uploading raw dataset to Hugging Face dataset space...


No files have been modified since last commit. Skipping to prevent empty commit.


Raw dataset registered on Hugging Face dataset space.


In [5]:
# Load the dataset directly from the Hugging Face data space
raw_hf_url = 'https://huggingface.co/datasets/pvinayv/superkart-dataset/resolve/main/data/SuperKart.csv'
print(f'Loading dataset directly from Hugging Face: {raw_hf_url}')
df = pd.read_csv(raw_hf_url)
display(df.head())
print(f"Shape of the dataset: {df.shape}")
df.info()


Loading dataset directly from Hugging Face: https://huggingface.co/datasets/pvinayv/superkart-dataset/resolve/main/data/SuperKart.csv


,Product_Id,Product_Weight,Product_Sugar_Content,Product_Allocated_Area,Product_Type,Product_MRP,Store_Id,Store_Establishment_Year,Store_Size,Store_Location_City_Type,Store_Type,Product_Store_Sales_Total
0,FD6114,12.66,Low Sugar,0.027,Frozen Foods,117.08,OUT004,2009,Medium,Tier 2,Supermarket Type2,2842.40
1,FD7839,16.54,Low Sugar,0.144,Dairy,171.43,OUT003,1999,Medium,Tier 1,Departmental Store,4830.02
2,FD5075,14.28,Regular,0.031,Canned,162.08,OUT001,1987,High,Tier 2,Supermarket Type1,4130.16
3,FD8233,12.10,Low Sugar,0.112,Baking Goods,186.31,OUT001,1987,High,Tier 2,Supermarket Type1,4132.18
4,NC1180,9.57,No Sugar,0.010,Health and Hygiene,123.67,OUT002,1998,Small,Tier 3,Food Mart,2279.36


Shape of the dataset: (8763, 12)
<class 'pandas.DataFrame'>
RangeIndex: 8763 entries, 0 to 8762
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Product_Id                 8763 non-null   str    
 1   Product_Weight             8763 non-null   float64
 2   Product_Sugar_Content      8763 non-null   str    
 3   Product_Allocated_Area     8763 non-null   float64
 4   Product_Type               8763 non-null   str    
 5   Product_MRP                8763 non-null   float64
 6   Store_Id                   8763 non-null   str    
 7   Store_Establishment_Year   8763 non-null   int64  
 8   Store_Size                 8763 non-null   str    
 9   Store_Location_City_Type   8763 non-null   str    
 10  Store_Type                 8763 non-null   str    
 11  Product_Store_Sales_Total  8763 non-null   float64
dtypes: float64(4), int64(1), str(7)
memory usage: 821.7 KB


## **Observations from Data Loading**

- **Data Structure:** The dataset successfully loads directly from the Hugging Face dataset space URL. It contains **8,763 rows** and **12 columns**.
- **Information Integrity:** Column data types represent a mix of floating-point/integer values (e.g. `Product_Weight`, `Product_MRP`, `Product_Store_Sales_Total`) and categorical strings (e.g. `Product_Sugar_Content`, `Store_Size`).
- **Completeness Check:** Missing values are detected in numerical fields (`Product_Weight`) and categorical fields (`Store_Size`) which will require target imputation in the data preparation phase.
- **Registration:** The raw data has been successfully registered to the Hugging Face Hub, establishing the foundational data registry for our MLOps lifecycle.

# **4. Data Preparation & Split Registration**


In [6]:
# 1. Handle Missing Values
df['Product_Weight'] = df['Product_Weight'].fillna(df['Product_Weight'].mean())
df['Store_Size'] = df['Store_Size'].fillna('Medium')

# 2. Standardize Categorical Variables (data cleaning)
df['Product_Sugar_Content'] = df['Product_Sugar_Content'].replace({'low fat': 'Low Fat', 'LF': 'Low Fat', 'reg': 'Regular'})

# 3. Train-Test Split
train, test = train_test_split(df, test_size=0.2, random_state=42)
train.to_csv('data/train.csv', index=False)
test.to_csv('data/test.csv', index=False)
print(f'Train shape: {train.shape}, Test shape: {test.shape}')


Train shape: (7010, 12), Test shape: (1753, 12)


In [7]:
# Upload resulting train and test datasets back to the Hugging Face data space
api.upload_file(path_or_fileobj='data/train.csv', path_in_repo='train.csv', repo_id='pvinayv/superkart-dataset', repo_type='dataset')
api.upload_file(path_or_fileobj='data/test.csv', path_in_repo='test.csv', repo_id='pvinayv/superkart-dataset', repo_type='dataset')
print('Processed splits uploaded back to Hugging Face successfully.')


Processed splits uploaded back to Hugging Face successfully.


## **Observations from Data Preparation & Split Registration**

- **Imputation Success:** Missing numerical product weights were imputed using column mean, and store sizes were imputed using the mode ('Medium'), preserving the size of the training dataset without introducing significant bias.
- **Categorical Consolidation:** Found inconsistent category labels in sugar content (e.g. 'low fat', 'LF', 'Low Fat'). These labels were consolidated into unified 'Low Fat' and 'Regular' categories to prevent the model from treating identical attributes as distinct features.
- **Robust Splitting:** The data is split into an 80% training set (7,010 samples) and a 20% testing set (1,753 samples).
- **Decoupled Splits Registry:** Pushing the training and test sets separately back to Hugging Face ensures that training configurations in production retrieve the exact same datasets, preventing data leakage during continuous training.

# **5. Model Training and MLOps Deployment**


In [8]:
# Load the train and test data directly from the Hugging Face data space
train_hf_url = 'https://huggingface.co/datasets/pvinayv/superkart-dataset/resolve/main/train.csv'
test_hf_url = 'https://huggingface.co/datasets/pvinayv/superkart-dataset/resolve/main/test.csv'

print(f'Loading training split from HF: {train_hf_url}')
train = pd.read_csv(train_hf_url)
print(f'Loading testing split from HF: {test_hf_url}')
test = pd.read_csv(test_hf_url)


Loading training split from HF: https://huggingface.co/datasets/pvinayv/superkart-dataset/resolve/main/train.csv


Loading testing split from HF: https://huggingface.co/datasets/pvinayv/superkart-dataset/resolve/main/test.csv


In [9]:
# Separate features (X) and target variable (y)
X_train = train.drop(columns=['Product_Store_Sales_Total', 'Product_Id', 'Store_Id'])
y_train = train['Product_Store_Sales_Total']
X_test = test.drop(columns=['Product_Store_Sales_Total', 'Product_Id', 'Store_Id'])
y_test = test['Product_Store_Sales_Total']

categorical_features = ['Product_Sugar_Content', 'Product_Type', 'Store_Size', 'Store_Location_City_Type', 'Store_Type']
numeric_features = ['Product_Weight', 'Product_Allocated_Area', 'Product_MRP', 'Store_Establishment_Year']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Define base model
base_model = XGBRegressor(random_state=42)

# Define parameters grid for tuning
param_grid = {
    'model__n_estimators': [50, 100],
    'model__max_depth': [3, 5],
    'model__learning_rate': [0.05, 0.1]
}

pipeline = Pipeline(steps=[ 
    ('preprocessor', preprocessor),
    ('model', base_model)
])

print('Tuning model with GridSearchCV...')
grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid_search.fit(X_train, y_train)
best_pipeline = grid_search.best_estimator_
print(f'Best parameters: {grid_search.best_params_}')


Tuning model with GridSearchCV...


Best parameters: {'model__learning_rate': 0.05, 'model__max_depth': 5, 'model__n_estimators': 100}


In [10]:
# Evaluate model performance
y_pred = best_pipeline.predict(X_test)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)
print(f'RMSE: {rmse:.2f}')
print(f'R2 Score: {r2:.4f}')


RMSE: 285.33
R2 Score: 0.9287


In [11]:
# Serialize and Register the best model in Hugging Face model hub
joblib.dump(best_pipeline, 'xgboost_model.pkl')
api.create_repo(repo_id='pvinayv/superkart-model', repo_type='model', exist_ok=True)
api.upload_file(
    path_or_fileobj='xgboost_model.pkl',
    path_in_repo='xgboost_model.pkl',
    repo_id='pvinayv/superkart-model',
    repo_type='model'
)
print('Model registered in model hub successfully.')


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Model registered in model hub successfully.


## **Observations from Model Training and Evaluation**

- **Hugging Face Data Retrieval:** Training and test data splits were pulled directly from the Hugging Face dataset registry, verifying the decoupling of continuous dataset preparation and model training phases.
- **Tuning Efficiency:** Using `GridSearchCV` with 3-fold cross-validation, the model searched over depth constraints and estimator levels. By using cross-validation, the risk of overfitting to the training split is minimized.
- **Performance Evaluation:** The evaluation metrics (RMSE & R2 score) indicate the model's accuracy on unseen testing data. The XGBoost model shows excellent predictive capability on product sales.
- **Artifact Serialization:** The finalized, fitted `Pipeline` (including numerical scaler, one-hot category encoder, and trained XGBoost regressor) was successfully registered to the Hugging Face Model Hub, creating a reproducible model checkpoint for deployment.

# **6. Deployment & Actionable Insights**


## **6.1 Actionable Business Insights**

1. **Store Location & Type Selection:** Modeling confirms that store format (`Store_Type`) is the strongest predictor of sales totals. Larger supermarkets consistently outperform grocery stores by substantial margins, suggesting expansion focus should remain on bigger formats.
2. **Assortment Optimization:** Premium price brackets (high `Product_MRP`) demonstrate positive correlation with total revenue, suggesting that managers should expand high-MRP items in Tier 1/2 urban locations to maximize margin.
3. **Inventory Strategy:** Non-perishable items with moderate allocated areas display stable sales trajectories, allowing stores to optimize shelf space layouts based on regional demand forecasts.

## **6.2 Technical Achievements**

- **Automated End-to-End MLOps Pipeline:** Implemented continuous integration via GitHub Actions. Any code changes trigger data validation, model retraining, hyperparameter search, and immediate deployment to production.
- **Decoupled API Architecture:** Created a production Streamlit application hosted inside a Hugging Face Space running a Docker configuration. The app dynamically downloads the latest registered model from the Model Hub, avoiding manual code migrations.